In [48]:
import os
import json
import re
import pandas as pd
from pathlib import Path
from PIL import Image
import time
import importlib
import gemini_helpers as gh
importlib.reload(gh)
from gemini_helpers import initialize_gemini, get_gemini_model, list_available_models

# --- 1. INITIALIZE GEMINI ---
# This uses our new helper function to set up the API key from the .env file
initialize_gemini()

# Show available models for this API key (that support generateContent)
try:
    avail = list_available_models()
    print(f"Available Gemini models (generateContent): {avail}")
except Exception as e:
    print(f"Could not list models: {e}")
    avail = []

# --- 1a. Choose a stable 2.5 model ---
preferred_name = "gemini-2.5-flash"
if preferred_name not in (avail or []):
    for cand in ["gemini-2.5-pro","gemini-2.5-flash-lite","gemini-flash-latest"]:
        if cand in (avail or []):
            preferred_name = cand
            break
print(f"Selecting model: {preferred_name}")
vision_model = get_gemini_model(preferred_name)

# --- 2. TASK/PLACE TAXONOMY ---
# Model should choose from these when possible
ALLOWED_TASK_TYPES = [
    "clean_table",
    "empty_trash",
    "organize_whiteboard",
    "tidy_windowsill",
    "check_shelf_area",
    "clean_floor",
    "close_window"
 ]
ALLOWED_PLACES = [
    "office","kitchen","classroom","bathroom","hallway","conference_room","lobby","break_room",
 ]

def normalize_choice(value: str, allowed: list[str], default: str = "other") -> str:
    if not isinstance(value, str):
        return default
    v = value.strip().lower()
    for opt in allowed:
        if v == opt.lower():
            return opt
    return default

# --- 3. AI MODEL AND PROMPTS (score-conditional) ---
# Score meaning: 0 = NOT DONE, 1 = PARTIALLY DONE, 2 = DONE
# Each prompt asks the model to return strict JSON only, containing:
#{
#  "task_type": one of ALLOWED_TASK_TYPES,
#  "place": one of ALLOWED_PLACES,
#  "captions": [4 strings matching the score condition]
#}

BASE_JSON_INSTRUCTIONS = f"""
Return ONLY a compact JSON object with keys: task_type, place, captions. Do not include markdown or code fences.
- task_type: Choose the single best label from: {ALLOWED_TASK_TYPES}.
- place: Choose the single best label from: {ALLOWED_PLACES}.
- captions: An array of exactly 4 short, natural sentences (10–25 words) matching the scene condition.
Output JSON only, no extra text."""

PROMPT_TEMPLATE_NOT_DONE = f"""You are labeling and describing an image for a cleaning-quality dataset.\n\nSCENE CONDITION: The area was NOT cleaned properly (task not done / failed cleaning).\nDescribe visible evidence of poor cleaning (clutter, spills, stains, dust, leftover items, writing not erased, overflowing bins).\n\n{BASE_JSON_INSTRUCTIONS}"""

PROMPT_TEMPLATE_PARTIALLY_DONE = f"""You are labeling and describing an image for a cleaning-quality dataset.\n\nSCENE CONDITION: The area was PARTIALLY cleaned — some effort is visible, but issues remain.\nDescribe typical partial-cleaning cues (streaks, missed spots, crumbs, smudges, water marks, areas still dirty).\n\n{BASE_JSON_INSTRUCTIONS}"""

PROMPT_TEMPLATE_DONE = f"""You are labeling and describing an image for a cleaning-quality dataset.\n\nSCENE CONDITION: The area is COMPLETELY cleaned — tidy, dry, free of debris, and orderly.\nDescribe clean/organized cues (dry surfaces, no stains, neatly arranged items, no residue).\n\n{BASE_JSON_INSTRUCTIONS}"""

PROMPT_BY_SCORE = {
    0: PROMPT_TEMPLATE_NOT_DONE,
    1: PROMPT_TEMPLATE_PARTIALLY_DONE,
    2: PROMPT_TEMPLATE_DONE,
}

def extract_json(text: str) -> str:
    """Try to extract a JSON object from raw model text (strip code fences/extra text)."""
    if not text:
        return ""
    # Remove code fences if present
    text = re.sub(r"^```[a-zA-Z]*\n|```$", "", text.strip())
    # Find braces region
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        return text[start:end+1]
    return text.strip()

def process_image(image_path_obj):
    image_path_str = str(image_path_obj.as_posix())
    try:
        score_folder = image_path_obj.parent.name
        score = int(score_folder) if score_folder.isdigit() else 0
        prompt = PROMPT_BY_SCORE.get(score, PROMPT_TEMPLATE_NOT_DONE)
        print(f"Processing: {image_path_str} (score={score})")
        img = Image.open(image_path_obj)
        response = vision_model.generate_content([prompt, img])
        raw = response.text or ""
        json_text = extract_json(raw)
        task_type = "other"
        place = "other"
        captions = []
        try:
            parsed = json.loads(json_text)
            task_type = normalize_choice(parsed.get("task_type"), ALLOWED_TASK_TYPES, "other")
            place = normalize_choice(parsed.get("place"), ALLOWED_PLACES, "other")
            caps = parsed.get("captions") or []
            if isinstance(caps, list):
                captions = [c for c in caps if isinstance(c, str)]
        except Exception:
            # Fallback: treat raw output as line-separated captions
            captions = [line.strip().lstrip("*- ") for line in raw.splitlines() if line.strip()]
        # Ensure 4 captions max
        captions = captions[:4]
        image_rows = []
        for caption in captions:
            clean_caption = caption.strip()
            if 5 < len(clean_caption) < 200:
                image_rows.append({
                    "image_path": image_path_str,
                    "score": score,
                    "description": clean_caption,
                    "task_type": task_type,
                    "place": place
                })
        time.sleep(0.4)
        return image_rows
    except Exception as e:
        print(f"!!! ERROR processing {image_path_str}: {e}")
        return []

def main(run_limit: int | None = None):
    base_dir = Path(".")
    image_paths = list(base_dir.rglob("dataset/train/**/*.jpg")) + list(base_dir.rglob("dataset/val/**/*.jpg"))
    print(f"Found {len(image_paths)} images to process.")
    all_data_rows = []
    for i, path in enumerate(image_paths):
        if run_limit is not None and i >= run_limit:
            print(f"Run limit {run_limit} reached; stopping early.")
            break
        rows = process_image(path)
        all_data_rows.extend(rows)
    print("\n--- Processing Complete ---")
    if not all_data_rows:
        print("No data was generated. Check your API key and image paths.")
        return
    output_file = "generated_dataset.csv"
    df = pd.DataFrame(all_data_rows)[["image_path","score","description","task_type","place"]]
    df.to_csv(output_file, index=False)
    print(f"Successfully saved {len(df)} rows to {output_file}")

# NOTE: main() intentionally not auto-invoked. Use: main(run_limit=10)

Available Gemini models (generateContent): ['gemini-2.5-pro-preview-03-25', 'gemini-2.5-flash-preview-05-20', 'gemini-2.5-flash', 'gemini-2.5-flash-lite-preview-06-17', 'gemini-2.5-pro-preview-05-06', 'gemini-2.5-pro-preview-06-05', 'gemini-2.5-pro', 'gemini-2.0-flash-exp', 'gemini-2.0-flash', 'gemini-2.0-flash-001', 'gemini-2.0-flash-lite-001', 'gemini-2.0-flash-lite', 'gemini-2.0-flash-lite-preview-02-05', 'gemini-2.0-flash-lite-preview', 'gemini-2.0-pro-exp', 'gemini-2.0-pro-exp-02-05', 'gemini-exp-1206', 'gemini-2.0-flash-thinking-exp-01-21', 'gemini-2.0-flash-thinking-exp', 'gemini-2.0-flash-thinking-exp-1219', 'gemini-2.5-flash-preview-tts', 'gemini-2.5-pro-preview-tts', 'learnlm-2.0-flash-experimental', 'gemma-3-1b-it', 'gemma-3-4b-it', 'gemma-3-12b-it', 'gemma-3-27b-it', 'gemma-3n-e4b-it', 'gemma-3n-e2b-it', 'gemini-flash-latest', 'gemini-flash-lite-latest', 'gemini-pro-latest', 'gemini-2.5-flash-lite', 'gemini-2.5-flash-image-preview', 'gemini-2.5-flash-image', 'gemini-2.5-fla

In [49]:
# --- TEST BLOCK: Process a single image ---
# This block helps debug issues by running the process on just one file.

print("--- Starting Single Image Test ---")
base_dir = Path(".")
test_image_path = None

# Let's find the first available image in the training set
image_extensions = ["*.jpg", "*.jpeg", "*.png"]
for ext in image_extensions:
    # Search within the '0', '1', '2' subdirectories
    possible_images = list(base_dir.rglob(f"dataset/train/**/{ext}"))
    if possible_images:
        test_image_path = possible_images[0]
        break

if test_image_path:
    print(f"Found test image: {test_image_path}")
    # Now, let's try to process it using your function
    # Make sure you have run the cell containing the 'process_image' function first
    try:
        single_image_rows = process_image(test_image_path)
        
        if single_image_rows:
            print("\n--- Test Successful! ---")
            print("Generated the following data for the test image:")
            # Print the first row as an example
            print(single_image_rows[0])
        else:
            print("\n--- Test Inconclusive ---")
            print("The 'process_image' function ran but returned no data.")
            print("This could be due to an API error (check the output above for '!!! ERROR') or an issue with the response from the AI.")

    except NameError:
        print("\n--- ERROR ---")
        print("The 'process_image' function is not defined. Please make sure to run the cell where it is defined before running this test cell.")
    except Exception as e:
        print(f"\n--- An Unexpected Error Occurred During the Test ---")
        print(f"Error: {e}")
else:
    print("\n--- Test Failed: No Images Found ---")
    print(f"Could not find any .jpg, .jpeg, or .png images in the 'dataset/train/' directory.")
    print("Please ensure your images are in the correct location (e.g., 'dataset/train/0/your_image.jpg').")

--- Starting Single Image Test ---
Found test image: dataset\train\0\PXL_20251107_192557855.MP.jpg
Processing: dataset/train/0/PXL_20251107_192557855.MP.jpg (score=0)

--- Test Successful! ---
Generated the following data for the test image:
{'image_path': 'dataset/train/0/PXL_20251107_192557855.MP.jpg', 'score': 0, 'description': 'The whiteboard is still covered with extensive blue writing, diagrams, and numbers from prior use.', 'task_type': 'organize_whiteboard', 'place': 'office'}

--- Test Successful! ---
Generated the following data for the test image:
{'image_path': 'dataset/train/0/PXL_20251107_192557855.MP.jpg', 'score': 0, 'description': 'The whiteboard is still covered with extensive blue writing, diagrams, and numbers from prior use.', 'task_type': 'organize_whiteboard', 'place': 'office'}


In [50]:
# Debug: print a compact list of models supporting generateContent
from gemini_helpers import initialize_gemini
import google.generativeai as genai

initialize_gemini()
count = 0
printed = []
try:
    for m in genai.list_models():
        methods = getattr(m, 'supported_generation_methods', []) or []
        name = getattr(m, 'name', '')
        short = name.split('/')[-1]
        if 'generateContent' in methods and 'gemini' in short:
            printed.append(short)
            count += 1
            if count >= 30:
                break
    print('Top generateContent models:', printed)
except Exception as e:
    print('Model listing failed:', e)


Top generateContent models: ['gemini-2.5-pro-preview-03-25', 'gemini-2.5-flash-preview-05-20', 'gemini-2.5-flash', 'gemini-2.5-flash-lite-preview-06-17', 'gemini-2.5-pro-preview-05-06', 'gemini-2.5-pro-preview-06-05', 'gemini-2.5-pro', 'gemini-2.0-flash-exp', 'gemini-2.0-flash', 'gemini-2.0-flash-001', 'gemini-2.0-flash-lite-001', 'gemini-2.0-flash-lite', 'gemini-2.0-flash-lite-preview-02-05', 'gemini-2.0-flash-lite-preview', 'gemini-2.0-pro-exp', 'gemini-2.0-pro-exp-02-05', 'gemini-exp-1206', 'gemini-2.0-flash-thinking-exp-01-21', 'gemini-2.0-flash-thinking-exp', 'gemini-2.0-flash-thinking-exp-1219', 'gemini-2.5-flash-preview-tts', 'gemini-2.5-pro-preview-tts', 'gemini-flash-latest', 'gemini-flash-lite-latest', 'gemini-pro-latest', 'gemini-2.5-flash-lite', 'gemini-2.5-flash-image-preview', 'gemini-2.5-flash-image', 'gemini-2.5-flash-preview-09-2025', 'gemini-2.5-flash-lite-preview-09-2025']


In [51]:
# --- SAMPLE RUN: generate ~127 rows ---
# Safe, small-batch run to validate end-to-end CSV creation.
main(run_limit=127)

Found 127 images to process.
Processing: dataset/train/0/PXL_20251107_192557855.MP.jpg (score=0)
Processing: dataset/train/0/PXL_20251107_192600830.jpg (score=0)
Processing: dataset/train/0/PXL_20251107_192600830.jpg (score=0)
Processing: dataset/train/0/PXL_20251107_212003382.MP.jpg (score=0)
Processing: dataset/train/0/PXL_20251107_212003382.MP.jpg (score=0)
Processing: dataset/train/0/PXL_20251107_212037468.jpg (score=0)
Processing: dataset/train/0/PXL_20251107_212037468.jpg (score=0)
Processing: dataset/train/0/PXL_20251107_212418130.jpg (score=0)
Processing: dataset/train/0/PXL_20251107_212418130.jpg (score=0)
Processing: dataset/train/0/PXL_20251107_212515209.jpg (score=0)
Processing: dataset/train/0/PXL_20251107_212515209.jpg (score=0)
Processing: dataset/train/0/PXL_20251107_212524767.jpg (score=0)
Processing: dataset/train/0/PXL_20251107_212524767.jpg (score=0)
Processing: dataset/train/0/PXL_20251107_212543077.jpg (score=0)
Processing: dataset/train/0/PXL_20251107_212543077.j